In [ ]:
%tensorflow_version 1.x

TensorFlow 1.x selected.


In [ ]:
pip install keras_layer_normalization

  Created wheel for keras-layer-normalization: filename=keras_layer_normalization-0.14.0-cp36-none-any.whl size=5268 sha256=765576de54471db2ad89acf6b2dd661750ed318a35fd5ace725f883380828c07
  Stored in directory: /root/.cache/pip/wheels/54/80/22/a638a7d406fd155e507aa33d703e3fa2612b9eb7bb4f4fe667
Successfully built keras-layer-normalization


In [ ]:
import os
import random
import scipy.io
import keras
import tensorflow as tf
from google.colab.patches import cv2_imshow
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

from keras.layers import Input, Conv2DTranspose, LSTM, ConvLSTM2D, BatchNormalization, TimeDistributed, Conv2D, Flatten, Dense, MaxPooling2D
from keras.models import Sequential, load_model
from keras_layer_normalization import LayerNormalization

drive.mount('/content/gdrive')   

Using TensorFlow backend.


Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/gdrive


In [ ]:
FUSION2020_FOLDER_PATH = "/content/gdrive/My Drive/FUSION2020"
NUMBER_OF_EXAMPLES_FOR_EACH_CLASS = 60
NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION = 15
EXPERIMENTS = 2
RANDOM_TRAINING_AND_VALIDATION_EXAMPLES = True

In [5]:

# Function to get the list of training examples paths
# inputs:
#   path:str -> Path to folder with .mat files.
#   number_of_examples:int -> Number of examples to load for each class. Will be loaded examples from 1 to number_of_examples+1
#   random_examples -> Do we select random examples?
# outputs:
#    If random_examples is false, it returns the first number_of_examples examples from each class. Else, number_of_examples random examples.

def get_training_and_validation_example_list(path, number_of_examples, random_examples):
  if random_examples:
    all_examples_paths = glob(os.path.join(path, "*.mat"))
    np.random.shuffle(all_examples_paths)
    training_matrix_paths = all_examples_paths[:number_of_examples]
  else:
    training_matrix_paths = []
    for i in range(1, number_of_examples+1):
      training_matrix_paths = training_matrix_paths + glob(os.path.join(path, "*_{}.mat".format(i)))
  return training_matrix_paths


# Function to get all examples matrix paths.
# inputs:
#   path:str -> Path to folder with .mat files.
# outputs:
#   Unsorted list of all .mat in path.

def get_all_matrix_paths(path):
  number_of_matrix_paths = len(glob(os.path.join(path, "*.mat")))
  all_matrix_paths = []
  for i in range(1, number_of_matrix_paths+1):
    all_matrix_paths = all_matrix_paths + glob(os.path.join(path, "*_{}.mat".format(i)))
  return all_matrix_paths

# Function to read all .mat files and return training and evaluation sets.
# inputs:
#   path:str -> Path to folder with .mat files.
#   number_of_training_examples:int -> Number of examples to use for training.
# outputs:
#   training data and evaluation data.

def read_all_matrix_from_object_folder_obtaining_training_and_evaluation(path, number_of_training_examples):
  all_matrix_paths = get_all_matrix_paths(path)                                 # We get all .mat paths.
  training_matrix_paths = get_training_and_validation_example_list(path,
                                                number_of_training_examples,
                                                RANDOM_TRAINING_AND_VALIDATION_EXAMPLES)    # We get the list of .mat paths we will use to train the network.
  print(training_matrix_paths)
  training_object_data = None                                                   # We initilize the training data matrix as None.
  evaluation_object_data = None                                                 # We initilize the evaluation data matrix as None.
  training_matrix_index = 0                                                     # We initialize the training matrix index as 0. This index will be used to insert training data into training_object_data.
  evaluation_matrix_index = 0                                                   # We initialize the evaluation matrix index as 0. This index will be used to insert training data into evaluation_object_data.

  for matrix_path in all_matrix_paths:                                          # For each .mat path...
    matrix = scipy.io.loadmat(matrix_path)                                        # We load the .mat as a matrix (numpy array).
    matrix = matrix["datos2"]                                                                
    print("Matrix {} shape".format(matrix_path))
    print(matrix.shape)
    if matrix_path in training_matrix_paths:                                      # If the matrix must be used as training data...
      print("Training data.")
      if training_object_data is None:                                              # If training_object_data has not been initialized as a matrix...
        training_object_data = -1 * np.ones((len(training_matrix_paths), 
                                             matrix.shape[0], matrix.shape[1]))       # We initialize training_object_data as a matrix with -1 values as undefined.
      training_object_data[training_matrix_index] = matrix                          # We insert the matrix into the first undefined training_object_data space defined by training_matrix_index.
      training_matrix_index += 1                                                    # We increase training_matrix_index.

    else:                                                                         # If the matrix mustn't be used as training data, it must be used as evaluation data...
      print("Evaluation data.")
      if evaluation_object_data is None:                                            # If evaluation_object_data has not been initialized as a matrix...
        evaluation_object_data = -1 * np.ones((
                                  (NUMBER_OF_EXAMPLES_FOR_EACH_CLASS
                                    -len(training_matrix_paths)),
                                  matrix.shape[0],
                                  matrix.shape[1]))                                   # We initialize evaluation_object_data as a matrix with -1 values as undefined.
      evaluation_object_data[evaluation_matrix_index] = matrix                      # We insert the matrix into the first undefined evaluation_object_data space defined by evaluation_matrix_index.
      evaluation_matrix_index += 1                                                  # We increase evaluation_matrix_index.

  print(training_object_data.shape)
  print(evaluation_object_data.shape)
  return training_object_data, evaluation_object_data                           # We return training and evaluation data.


# Function to get ordered training and evaluation objects.
# inputs:
#   path:str -> Folder with folders that contains .mats
#   number_of_training_examples:int -> Number of examples used as training and evaluation
# outputs:
#   training data, training labels, evaluation data and evaluation labels.

def read_training_and_evaluation_objects_data(path,number_of_training_examples):
  nom_labels = ["ball_l_", "basket_ball_", "coke_bottle_", "energy_drink_can_", 
                "gears_box_", "grey_mouse_", "mixed_nut_", "mixed_washer_", 
                "mouse_", "nestea_bottle_", "nut_m6_", "nut_m8_", "nut_m10_", 
                "rivets_box_", "roll_wheel_", "rubber_pipe_", "soda_can_", 
                "sponge_rough_", "sponge_rough_inclusions_", "sponge_scrunchy_",
                "sponge_smooth_", "sponge_smooth_inclusions_", "sponge_tube_", 
                "water_bottle_"]                                                # We define all labels (classes) names that match with subfolders names.
  all_objects_folders_path = []                                                 # We initialize the list with folders paths.
  for nom_label in nom_labels:                                                  # For each label name...
    all_objects_folders_path = all_objects_folders_path + glob(
                                                os.path.join(path,nom_label))     # We concatenate the subfolder path with that label to the all_objects_folders_path list.

  all_objects_training_data = None                                              # We initialize the variable to contain training data (all_objects_training_data) as None.
  all_objects_evaluation_data = None                                            # We initialize the variable to contain evaluation data (all_objects_evaluation_data) as None.
  all_objects_training_labels = []                                              # We initialize the training data labels list (all_objects_training_labels) as None.
  all_objects_evaluation_labels = []                                            # We initialize the evaluation data labels list (all_objects_evaluation_labels) as None.

  for object_index, object_folder_path in enumerate(all_objects_folders_path):  # For each object class subfolder path and its index...
    #print(object_folder_path)
    training_object_data, evaluation_object_data = \
      read_all_matrix_from_object_folder_obtaining_training_and_evaluation(
                                    object_folder_path,
                                    number_of_training_examples)                  # We get the training and evaluation objects data for that class.

    if all_objects_training_data is None:                                         # If all_objects_training_data has not been initialized as a list...
      all_objects_training_data = training_object_data                              # We initialize all_objects_training_data as the current training_object_data as initial list.
    else:                                                                         # else...
      all_objects_training_data = np.concatenate((all_objects_training_data, 
                                                  training_object_data),
                                                axis = 0)                           # We concatenate current training_object_data list to all_objects_training_data.

    if all_objects_evaluation_data is None:                                       # If all_objects_evaluation_data has not been initialized as a list...
      all_objects_evaluation_data = evaluation_object_data                          # We initialize all_objects_evaluation_data as the current evaluation_object_data as initial list.
    else:                                                                         # else...
      all_objects_evaluation_data = np.concatenate((all_objects_evaluation_data, 
                                                    evaluation_object_data), 
                                                  axis = 0)                         # We concatenate current evaluation_object_data list to all_objects_evaluation_data.

    all_objects_training_labels = all_objects_training_labels + \
                                    number_of_training_examples*[object_index]    # We concatenate number_of_training_examples index values to all_objects_training_labels in order to have objects labels.
    
    number_of_evaluation_examples = (NUMBER_OF_EXAMPLES_FOR_EACH_CLASS
                                      -number_of_training_examples)               # We get the number of evaluation examples.

    all_objects_evaluation_labels = all_objects_evaluation_labels + \
                              number_of_evaluation_examples*[object_index]        # We concatenate number of evaluation examples index values to all_objects_training_labels in order to have objects labels.

  return (all_objects_training_data, np.array(all_objects_training_labels), 
          all_objects_evaluation_data, np.array(all_objects_evaluation_labels))  # We return training data, training labels, evaluation data and evaluation labels.

#Function to print and save information
def print_and_save_information(history, experiment_index):
  # list all data in history
  print(history.history.keys())
  # summarize history for accuracy
  plt.plot(history.history['accuracy'])
  plt.plot(history.history['val_accuracy'])
  plt.title('model accuracy')
  plt.ylabel('accuracy')
  plt.xlabel('epoch')
  plt.legend(['train', 'test'], loc='upper left')
  if RANDOM_TRAINING_AND_VALIDATION_EXAMPLES:
    plt.savefig(os.path.join(FUSION2020_FOLDER_PATH,"review/"+str(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION)+"/model_accuracy_LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.png".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION, experiment_index)))
  else:
    plt.savefig(os.path.join(FUSION2020_FOLDER_PATH,"review/model_accuracy_LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.png".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION, experiment_index)))
  plt.show()
  # summarize history for loss
  plt.plot(history.history['loss'])
  plt.plot(history.history['val_loss'])
  plt.title('model loss')
  plt.ylabel('loss')
  plt.xlabel('epoch')
  plt.legend(['train', 'test'], loc='upper left')
  if RANDOM_TRAINING_AND_VALIDATION_EXAMPLES:
    plt.savefig(os.path.join(FUSION2020_FOLDER_PATH,"review/"+str(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION)+"/model_loss_LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.png".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION, experiment_index)))
  else:
    plt.savefig(os.path.join(FUSION2020_FOLDER_PATH,"review/model_loss_LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.png".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION, experiment_index)))
  plt.show()

In [6]:
# Function to define and train the classification network model.

def train_model(training_data, training_labels):
  seq = Sequential()
  # # # # #
  seq.add(LSTM(1000, input_shape=(4,41), return_sequences=True, activation = "tanh"))
  seq.add(LSTM(100, return_sequences=False, activation = "tanh"))
  # # # # #
  #seq.add(Flatten())
  seq.add(Dense(24, activation = "softmax"))
  print(seq.summary())

  seq.compile(optimizer=keras.optimizers.Adam(lr=0.00001),
              loss="categorical_crossentropy",
              metrics=['accuracy'])
  history = seq.fit(training_data, training_labels, batch_size = 16, epochs=1000, validation_split = 0.2, shuffle = True)
  return seq, history

In [7]:
for experiment_index in range(EXPERIMENTS):
  print("EXPERIMENT {}.".format(experiment_index))
  print("Loading data.")
  #examples, labels = read_objects_data(os.path.join(FUSION2020_FOLDER_PATH, "Datos-arduino"))
  all_objects_training_data, all_objects_training_labels, all_objects_evaluation_data, all_objects_evaluation_labels = read_training_and_evaluation_objects_data(os.path.join(FUSION2020_FOLDER_PATH, "Datos-arduino_Norm"), NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION)
  all_objects_training_labels = keras.utils.to_categorical(all_objects_training_labels)
  all_objects_evaluation_labels = keras.utils.to_categorical(all_objects_evaluation_labels)
  print(all_objects_training_data.shape)
  print(all_objects_training_labels.shape)
  print(all_objects_evaluation_data.shape)
  print(all_objects_evaluation_labels.shape)
  #read_objects_data(os.path.join(FUSION2020_FOLDER_PATH, "DatasetIEEESensors3DCNN"))
  reshaped_all_objects_training_data = all_objects_training_data
  reshaped_all_objects_training_data = list(reshaped_all_objects_training_data)
  all_objects_training_labels = list(all_objects_training_labels)
  zip_list = list(zip(reshaped_all_objects_training_data, all_objects_training_labels))
  random.shuffle(zip_list)
  reshaped_all_objects_training_data, all_objects_training_labels = zip(*zip_list)
  reshaped_all_objects_training_data = np.array(reshaped_all_objects_training_data)
  all_objects_training_labels = np.array(all_objects_training_labels)
  print(reshaped_all_objects_training_data.shape)

  print(np.min(reshaped_all_objects_training_data))
  print(np.max(reshaped_all_objects_training_data))
  model, history = train_model(reshaped_all_objects_training_data, all_objects_training_labels)
  print(all_objects_training_labels[9])
  reshaped_all_objects_evaluation_data = all_objects_evaluation_data

  score = model.evaluate(reshaped_all_objects_evaluation_data, all_objects_evaluation_labels)
  print('Test loss: {} / Test accuracy: {}'.format(score[0], score[1]))
  if RANDOM_TRAINING_AND_VALIDATION_EXAMPLES:
    output_file = os.path.join(FUSION2020_FOLDER_PATH,"review/"+str(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION)+"/LSTMpythonArduino_review_{}_examples_training_and_validation_test_scores.txt".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION))
  else:
    output_file = os.path.join(FUSION2020_FOLDER_PATH,'review/LSTMpythonArduino_review_{}_examples_training_and_validation_test_scores.txt'.format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION))
  
  with open(output_file, 'a') as f:
    print('Test loss: {} / Test accuracy: {}'.format(score[0], score[1]), file=f)

  evaluation_predictions = model.predict(reshaped_all_objects_evaluation_data)
  print(evaluation_predictions.shape)

  output_matrix = -1 * np.ones((evaluation_predictions.shape[1],evaluation_predictions.shape[0]))

  for i in range(reshaped_all_objects_evaluation_data.shape[0]):
    output_matrix[:,i] = evaluation_predictions[i]
  if RANDOM_TRAINING_AND_VALIDATION_EXAMPLES:
    scipy.io.savemat(os.path.join(FUSION2020_FOLDER_PATH,"review/"+str(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION)+"/LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.mat".format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION,experiment_index)), {'output':output_matrix})
  else:
    scipy.io.savemat(os.path.join(FUSION2020_FOLDER_PATH,'review/LSTMpythonArduino_review_{}_examples_training_and_validation_{}_experiment.mat'.format(NUMBER_OF_EXAMPLES_FOR_TRAINING_AND_VALIDATION,experiment_index)), {'output':output_matrix})

Se han truncado las últimas 5000 líneas del flujo de salida.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_47.mat shape
(4, 41)
Evaluation data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_48.mat shape
(4, 41)
Evaluation data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_49.mat shape
(4, 41)
Evaluation data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_50.mat shape
(4, 41)
Training data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_51.mat shape
(4, 41)
Training data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_52.mat shape
(4, 41)
Evaluation data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-arduino_Norm/mixed_washer_/mixed_washer_53.mat shape
(4, 41)
Evaluation data.
Matrix /content/gdrive/My Drive/FUSION2020/Datos-ardui

In [ ]:
example_to_print = 10
print(all_objects_training_data)
print(np.max(all_objects_training_data))
print(np.min(all_objects_training_data))

[[[0.53681633 0.53681633 0.53681633 ... 0.62302041 0.62302041 0.62302041]
  [0.54187755 0.54187755 0.54187755 ... 0.67861224 0.67861224 0.67861224]
  [0.16595918 0.16595918 0.16595918 ... 0.44538776 0.44538776 0.44538776]
  [0.1924898  0.1924898  0.1924898  ... 0.43534694 0.43534694 0.43534694]]

 [[0.59510204 0.59510204 0.59510204 ... 0.69142857 0.69142857 0.69142857]
  [0.56865306 0.56865306 0.56865306 ... 0.70538776 0.70538776 0.70538776]
  [0.15012245 0.15012245 0.15012245 ... 0.48342857 0.48342857 0.48342857]
  [0.23134694 0.23134694 0.23134694 ... 0.54163265 0.54163265 0.54163265]]

 [[0.59257143 0.59257143 0.59510204 ... 0.66604082 0.66604082 0.67363265]
  [0.60620408 0.6035102  0.60081633 ... 0.73493878 0.74293878 0.7402449 ]
  [0.20187755 0.19967347 0.19967347 ... 0.50791837 0.52008163 0.52293878]
  [0.19183673 0.19183673 0.19183673 ... 0.43820408 0.43893878 0.47918367]]

 ...

 [[0.70155102 0.70155102 0.70155102 ... 0.75216327 0.74963265 0.75216327]
  [0.63036735 0.63036735 0

In [ ]:
print(all_objects_training_labels[9])



[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
